In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

import os 
from dotenv import load_dotenv
# This function will load all the variables from .env file into the environment
# make them available to in the os.environ dictionary
load_dotenv()   

if os.environ.get("GOOGLE_API_KEY"):
    print ("GOOGLE_API_KEY found in the environment variables")
else:
    raise ValueError ("GOOGLE_API_KEY not found in the environment variables")


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# for the string out praser 
from langchain_core.output_parsers import StrOutputParser ,PydanticOutputParser


llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

GOOGLE_API_KEY found in the environment variables


# **Conditional chains**
case : 
 In the product review . you do not need to build you NLP or Natural language processing and machine learning model to categorize it. you can leverage LLM. That's why you are an AI data engineer we don't invoke an ML engineer.
 Step : 
   now will take the review and we will pass it to LLM and the LLM actually categorize it whether the review is positive or negative . 
   LLm here are actually the NLP models and Behind the scenes built on the top of the transformer model . so they are well trained on these kind of tasks. 

   LLm are very well trained on the kinds of task like categorization and evaluate because they 
   trained one large data

Guide : 
    we need to guide LLM . we need to also use a structured output  with pydantic because LLM yes can do that work . but LLM will not just say yes or no . you need to guide LLM as reponsed as yes or no . do not say other information yes or no with pydantic.  

Advanatages of using the Premium models which are hosted by the reputable providers like OpenAI, Anthropic, and Cohere. becuase they understand everything and we do not need to do anything  : 

demo_chain = llm_structured_output | PydanticOutputParser(pydantic_model=llm_schema)

# **PydanticOutputParser(pydantic_model=llm_schema)** we remove this because you do not need to this 
# Reason :
 because you  are already converting the text into a pydantic model if we see the response
# it is already there => 
llm_schema(movie_summary_flag='Positive') if you LLM would be just generating the dictionary  but not a pydantic then we have to use pydanticoutputparser




In [18]:
# to understand the structured output  
from pydantic import BaseModel 
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag:Literal["Positive","Negative"] # liternal means it can only take these values  

llm_structured_output =llm_gemini.with_structured_output(llm_schema)
 # this will guide the LLM to output the structured data as per the schema

demo_chain = llm_structured_output 
# PydanticOutputParser(pydantic_model=llm_schema) we remove this because you do not need to this 
#Reason : because you  are already converting the text into a pydantic model if we see the response
#it is already there => llm_schema(movie_summary_flag='Positive') if you LLM would be just generating the dictionary  but not a pydantic then we have to use pydanticoutputparser

demo_chain.invoke("the movie was great")

#PydanticOutputParser is a class that helps to parse the output of the LLM and 
#why we need this? is that we need to guide the LLM to output the structured data as per the schema 

llm_schema(movie_summary_flag='Positive')

In [19]:
# to code the above 
from pydantic import BaseModel 
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag:Literal["Positive","Negative"] # liternal means it can only take these values  

llm_structured_output =llm_gemini.with_structured_output(llm_schema)

In [20]:
# this cell explain detail about the concept we used in the custom runnable 

result = llm_structured_output.invoke("This movie is great")
result # ouptut is llm_schema(movie_summary_flag='Positive') is an pydantic object 
result.model_dump() # this conver the pydantic object to a dictionary as this {'movie_summary_flag': 'Positive'}

#.model_dump() is  specially with the pydantic object 

# if i just want to get the movie_summary_flag i will use like this 
result.model_dump()['movie_summary_flag'] # the output is Positive we get only the flag by this  

'Positive'

# **Chain with conditional Chain**    

 in this prompt_template ask the LLM to movie review  or just ask LLM to write a quick movie review  about a movie as evaluator.

In [21]:
# TASK -1 [prompt]
Prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie reviewer evaluator"),
    ("human", " Please categorize the review as positive or negative : {input}") # this will reduce my token because i mention how my ouptut i need .
])

In [22]:
# Task -2 [LLM]
llm_structured_output =llm_gemini.with_structured_output(llm_schema)

In [23]:
# Task -3 [str Parser]

#Str_parser = StrOutputParser() we to not want the str output parser because we have the pydantic model   


In [24]:
# Task -3 [custom Runnable]
# Here we need to use the custom Runnable to covert the pydantic object to kind of dictionary 
# In above we will receive the pydantic object and we need to convert it to kind of  dictionary 
# In order to create the furthur contidinal we need to use kind of customer function 

#here we want a JSON   
from langchain_core.runnables import RunnableLambda
 
def pydantic_json(input:llm_schema) -> str : # input is the pydantic object and llm_schema is the pydantic object both are same here to convert the string.
  return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

# **Conditional Chain 1**

**Core** :In the above chain we ask the LLM to create movie reviwer next the chain 1 we will create a linkedln post for the movie and for the chain 2 we will create instragram post for movie

In [25]:
# Task -1 [Prompt]

linkedin_prompt=ChatPromptTemplate.from_messages([
    ("system"," you are a linkedin post generator"),
    ("human","create a post for the following text for Linkedkin :{text}")
])
#Task -2 [LLM]
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

#Task -3[Str praser]
str_praser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_gemini |str_praser # instead of creating chain like this we 
# we create a function 

# **Conditional Chain 2** 

In [26]:
from langchain_core.runnables import RunnableParallel , RunnableLambda , RunnableBranch

In [33]:
"""def insta_chain(text:dict):
    text=text["text"]
    # Task -1 [Prompt]

    insta_prompt=ChatPromptTemplate.from_messages([
        ("system"," you are a instagram post generator"),
        ("human","create a post for the following text for instagram :{text}")
    ])
    #Task -2 [LLM]
    llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

    #Task -3[Str praser]
    str_praser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_praser
    result = chain_insta.invoke(text)
    return result"""

def insta_chain(text: str):   # 👈 accept string, not dict
    # IMPORTANT:
# This chain receives a STRING ("Positive" / "Negative") from the previous step
# after pydantic_json_lambda.
# DO NOT treat the input as a dictionary (text["text"]) here,
# otherwise it will raise:
# TypeError: string indices must be integers, not 'str'
#
# Always wrap the string into {"text": value} only when invoking the prompt.
    # Task -1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "you are a instagram post generator"),
        ("human", "create a post for the following text for instagram : {text}")
    ])

    llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
    str_praser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_praser
    return chain_insta.invoke({"text": text})   # 👈 wrap here

     

# both concept of parallel chain ar same but small difference is that first we create chain and then we invoke it 
# with second option we do same but in function  format on top of it 
# it is easy to manage because you can import this function in other file and use it there and import this function as a utility function

# second Benefit : 
   #you can make the changes in between tasks because if you do not like we easy change the prompt or llm or parser 
   #if we create function like this you do not need those function like dictionary_marker because we can make the change in line 

insta_chain_runnnable = RunnableLambda(insta_chain)
# RunnableLambda is a class that takes a function as an argument and returns a runnable object by that we can use it as a chain 

we need to finalize the final orchestrator . that means we need to now arrange task one , task two and task three , task four and then these two conditional chains with conditional chain 1 and Parallel chain 2 .   

# **final orchestrator** 

In [34]:
# runnableBranch we need to import that as runnableBranch


conditional_chains =RunnableBranch(

    # here we will define all the branch 
    # usually we simple write a function directly . because we want to go to this chain 1 is yes as input and the chain for no for that we
    # to define a function . now we doing to use the lambda function .
   (lambda x: "positive" in x ,chain_linkedin),
   #(lambda x: "negative" in x ,insta_chain_runnnable), we are remove this and change this as default chain .
  # this two conditions we have created . 
    #in this we have set one as the default chain now because we have not created any default chain .
   insta_chain_runnnable #here this will acta default chain because we have not created any default chain . 
) 

final_orchestration = Prompt_template | llm_structured_output | pydantic_json_lambda |conditional_chains 


In [35]:
final_orchestration.invoke({"input": "I loved this KGF movie"})

'Okay, I\'m ready to generate an Instagram post based on the theme "Positive." To give you the best options, I need a little more information.  Please tell me:\n\n*   **What kind of positive message do you want to convey?** (e.g., self-love, gratitude, overcoming challenges, inspiring others, a general good vibe, etc.)\n*   **Who is your target audience?** (e.g., young adults, entrepreneurs, parents, etc.)\n*   **Do you have a specific image or video you want to use?** (If so, describe it or provide a link.)\n*   **Do you have any specific hashtags you want to include?**\n*   **What is the overall tone you are going for?** (e.g., motivational, lighthearted, reflective, etc.)\n\nIn the meantime, here are a few general options to get you started:\n\n**Option 1: Simple & Sweet**\n\n*   **Image:** A bright, cheerful photo (e.g., a sunrise, flowers, a smiling face).\n*   **Caption:** "Choosing positivity today and every day! ✨ What\'s one thing that\'s making you smile? Share in the comment